In [1]:
import pandas as pd
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split


In [ ]:
import pandas as pd
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split

print("Torch Device:", "CPU")

# ========================
# SETTINGS (SPEED BOOST 🔥)
# ========================
IMG_SIZE = 32
BATCH_SIZE = 64
EPOCHS = 5
NUM_CLASSES = 43

# =========================
# LOAD CSV
# =========================
train_csv_path = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB\Train.csv"
test_csv_path  = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB\Test.csv"
base_path      = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB"

train_df = pd.read_csv(train_csv_path)

# 🔥 Speed Boost – use only 50% data
train_df = train_df.sample(frac=0.5, random_state=42)


# =========================
# DATASET CLASS
# =========================
class TrafficDataset(Dataset):
    def __init__(self, df, base, transform=None):
        self.df = df
        self.base = base
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.base + "\\" + row["Path"]
        img = cv2.imread(img_path)

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        label = int(row["ClassId"])
        return img, label


# =========================
# TRANSFORMS
# =========================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["ClassId"])

train_dataset = TrafficDataset(train_df, base_path, transform)
val_dataset   = TrafficDataset(val_df, base_path, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# =========================
# MODEL (SUPER FAST 🚀)
# =========================
model = models.mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


# =========================
# TRAINING
# =========================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

    # =========================
    # VALIDATION
    # =========================
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum()

    acc = 100 * correct / total
    print("Validation Accuracy:", acc, "%")


# =========================
# SAVE MODEL
# =========================
torch.save(model.state_dict(), "traffic_fast_model.pth")
print("Model Saved Successfully 😎🔥")


Torch Device: CPU


c:\Users\hp\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# datagen=ImageDataGenerator()

# data= datagen.flow_from_directory('GTRB', target_size=(64,64))
# # 

In [ ]:
# pip install torch torchvision torchaudio


In [ ]:
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.9.1+cpu
False


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 



In [ ]:
import torch 
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

# Define image transrormations
transform= transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# load the dataset 
train_dataset = datasets.ImageFolder(root='GTRB', transform=transform)

# Create the trainloder
trainloader= DataLoader(train_dataset,batch_size=32,shuffle=True)


In [ ]:
print(f'Dataset loaded with {len(train_dataset)} images.')
print(f"Classes found: {train_dataset.classes}")

Dataset loaded with 51882 images.
Classes found: ['Meta', 'Test', 'Train']


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN,self).__init__()
        self.conv1=nn.Conv2d(3,32,kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(32* 32 * 32, 128)
        self.fc2 = nn.Linear(128,3)


    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x= x.view(-1, 32 * 32*32)
        x=F.relu(self.fc1(x))
        x=self.fc2(x) 
        return x
    

model= SimpleCNN()

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

In [ ]:
import torch.optim as optim

criterion= nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

for epoch in range(10):
    running_loss=0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        #forward pass
        outputs = model(images)
        loss= criterion(outputs,labels)
        
        #bakward pass
        loss.backward()
        optimizer.step()

        running_loss +=loss.item()

    print(f"Epoch {epoch+1}/10 complete, Loss: {running_loss/len(trainloader): .4f}")

KeyboardInterrupt: 